# CELL 1
# Feature Engineering

Cleaning, imputation, feature design, and the train/test split — everything between raw data and a model-ready dataset.

| | |
|---|---|
| **Notebook** | 03 of 05 |
| **Previous** | `02_portfolio_analysis.ipynb` |
| **Next** | `04_modeling.ipynb` |

---

## Goal

Notebook 02 found that grade, purpose, geography, DTI, and vintage all carry independent risk signal. This notebook turns those findings into features the model can learn from.

The output is a clean, leakage-free, train/test split ready for the three classifiers in notebook 04.

## Sections

| # | Focus |
|---|---|
| 1 | Setup and raw data load |
| 2 | Apply column drop list, filter to terminal loans |
| 3 | Type cleaning — percentages, dates, employment length |
| 4 | Missing value strategy |
| 5 | Engineered features (the analytical judgment lives here) |
| 6 | Categorical encoding |
| 7 | Stratified train/test split |
| 8 | Persist datasets for notebook 04 |
| 9 | Handoff |

# CELL 2
## 1. Setup and raw data load

Standard preamble: imports, palette, the same 500K random sample as notebooks 01 and 02. Notebook 03 also loads the column drop list that notebook 02 produced (`outputs/columns_to_drop.json`) — so anything dropped here is justified by the audit in section 8 of the previous notebook, not by ad-hoc decisions.

In [1]:
# CELL 3
# ─────────────────────────────────────────────────────────────────────────────
#  Imports and display configuration
# ─────────────────────────────────────────────────────────────────────────────
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

os.makedirs('outputs', exist_ok=True)

PALETTE = {
    'bg':      '#0f1117',
    'panel':   '#1a1d29',
    'text':    '#e8eaed',
    'muted':   '#8b8fa3',
    'primary': '#4c9aff',
    'success': '#36b37e',
    'danger':  '#ff5630',
    'warn':    '#ffab00',
    'accent':  '#8777d9',
}

plt.rcParams.update({
    'figure.facecolor':  PALETTE['bg'],
    'axes.facecolor':    PALETTE['bg'],
    'savefig.facecolor': PALETTE['bg'],
    'axes.edgecolor':    PALETTE['muted'],
    'axes.labelcolor':   PALETTE['text'],
    'text.color':        PALETTE['text'],
    'xtick.color':       PALETTE['muted'],
    'ytick.color':       PALETTE['muted'],
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.titlesize':    14,
    'axes.titleweight':  'bold',
    'axes.titlepad':     18,
    'figure.figsize':    (12, 6),
    'figure.dpi':        110,
})

In [2]:
# CELL 4
# ─────────────────────────────────────────────────────────────────────────────
#  Load the same 500K sample, then apply notebook 02's column drop list
# ─────────────────────────────────────────────────────────────────────────────
SAMPLE_SIZE = 500_000
SEED        = 42

total_rows = sum(1 for _ in open('accepted_2007_to_2018Q4.csv')) - 1
rng        = np.random.default_rng(SEED)
keep_idx   = set(rng.choice(total_rows, size=SAMPLE_SIZE, replace=False))
skip_rows  = [i + 1 for i in range(total_rows) if i not in keep_idx]

df = pd.read_csv(
    'accepted_2007_to_2018Q4.csv',
    low_memory=False,
    skiprows=skip_rows,
)

# Apply the drop list produced by notebook 02
with open('outputs/columns_to_drop.json') as f:
    drop_groups = json.load(f)

cols_to_drop = list(set(drop_groups['sparse'] + drop_groups['leakage'] + drop_groups['junk']))
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print(f"Loaded sample          : {SAMPLE_SIZE:,} rows")
print(f"Columns dropped        : {len(cols_to_drop)}")
print(f"  ├─ sparse (>50% NA)  : {len(drop_groups['sparse'])}")
print(f"  ├─ leakage           : {len(drop_groups['leakage'])}")
print(f"  └─ identifiers/junk  : {len(drop_groups['junk'])}")
print(f"Columns remaining      : {df.shape[1]}")

Loaded sample          : 500,000 rows
Columns dropped        : 68
  ├─ sparse (>50% NA)  : 44
  ├─ leakage           : 23
  └─ identifiers/junk  : 10
Columns remaining      : 83


# CELL 5
## 2. Filter to terminal loans

The model needs labeled outcomes. Loans still in `Current`, `Late`, or `In Grace Period` haven't reached a final state — we don't know if they'll end up paid or written off. Including them would either require excluding them from the target (wasted rows) or labeling them ambiguously (noisy signal).

So: filter to `Fully Paid` and `Charged Off` only. Build the binary target. Drop the original `loan_status` to avoid accidental leakage.

In [3]:
# CELL 6
# ─────────────────────────────────────────────────────────────────────────────
#  Restrict to terminal loans, build binary target
# ─────────────────────────────────────────────────────────────────────────────
before = len(df)

df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df['default'] = (df['loan_status'] == 'Charged Off').astype(int)
df = df.drop(columns=['loan_status'])

after = len(df)

print(f"Rows before filter   : {before:>10,}")
print(f"Rows after filter    : {after:>10,}")
print(f"Dropped (active/late): {before - after:>10,}")
print(f"Default rate         : {df['default'].mean():>10.1%}")
print(f"Default count        : {df['default'].sum():>10,}")
print(f"Paid count           : {(df['default'] == 0).sum():>10,}")

Rows before filter   :    500,000
Rows after filter    :    297,299
Dropped (active/late):    202,701
Default rate         :      20.0%
Default count        :     59,338
Paid count           :    237,961


# CELL 7
## 3. Type cleaning

Several columns arrive from the CSV in formats that aren't directly usable:

| Column | Format issue | Fix |
|---|---|---|
| `int_rate` | String like `"13.49%"` | Strip `%`, cast to float |
| `revol_util` | String like `"45.20%"` | Strip `%`, cast to float |
| `term` | String like `" 36 months"` | Extract integer 36 |
| `emp_length` | String like `"10+ years"`, `"< 1 year"` | Map to integer years |
| `issue_d` | String like `"Mar-2015"` | Parse to datetime, extract year |
| `earliest_cr_line` | Same as above | Parse, then derive credit-history-length feature |

Each fix is small. Together they take six columns from "unusable strings" to "model-ready numerics."

In [4]:
# CELL 8
# ─────────────────────────────────────────────────────────────────────────────
#  Type cleaning — six columns that arrive as strings
# ─────────────────────────────────────────────────────────────────────────────

# (a) Percentages — strip "%" and cast to float
for col in ['int_rate', 'revol_util']:
    if col in df.columns:
        df[col] = (
            df[col].astype(str)
                   .str.rstrip('%')
                   .replace({'nan': np.nan})
                   .astype(float)
        )

# (b) Term — extract integer months from "36 months" / "60 months"
df['term'] = df['term'].astype(str).str.extract(r'(\d+)').astype(float)

# (c) Employment length — map text categories to integer years
emp_length_map = {
    '< 1 year': 0,  '1 year':   1,  '2 years':  2,  '3 years':  3,
    '4 years':  4,  '5 years':  5,  '6 years':  6,  '7 years':  7,
    '8 years':  8,  '9 years':  9,  '10+ years': 10,
}
df['emp_length'] = df['emp_length'].map(emp_length_map)

# (d) Dates — parse both date columns
df['issue_d']           = pd.to_datetime(df['issue_d'],          format='%b-%Y', errors='coerce')
df['earliest_cr_line']  = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')

# (e) Pull issue year out as its own column for vintage-aware splitting later
df['issue_year'] = df['issue_d'].dt.year

# ─── Sanity check ───────────────────────────────────────────────────────────
type_check = pd.DataFrame({
    'column':    ['int_rate', 'revol_util', 'term', 'emp_length', 'issue_d', 'earliest_cr_line', 'issue_year'],
    'dtype':     [str(df[c].dtype) for c in
                 ['int_rate', 'revol_util', 'term', 'emp_length', 'issue_d', 'earliest_cr_line', 'issue_year']],
    'sample':    [df[c].dropna().iloc[0] for c in
                 ['int_rate', 'revol_util', 'term', 'emp_length', 'issue_d', 'earliest_cr_line', 'issue_year']],
    'missing %': [(df[c].isna().mean() * 100).round(2) for c in
                 ['int_rate', 'revol_util', 'term', 'emp_length', 'issue_d', 'earliest_cr_line', 'issue_year']],
})

print("Type-cleaning result:\n")
print(type_check.to_string(index=False))

Type-cleaning result:

          column          dtype              sample  missing %
        int_rate        float64               11.99       0.00
      revol_util        float64               19.20       0.06
            term        float64               36.00       0.00
      emp_length        float64               10.00       5.85
         issue_d datetime64[us] 2015-12-01 00:00:00       0.00
earliest_cr_line datetime64[us] 1999-12-01 00:00:00       0.00
      issue_year          int32                2015       0.00


# CELL 9
## 4. Engineered features

Section 4 of notebook 02 found three numeric variables with strong default signal: DTI, income, and loan size. Section 7 found purpose carries large independent risk variance. Sections 2–3 found grade and sub-grade encode coarse risk.

This section turns those raw signals into engineered features the model can use directly:

| Feature | Type | Why it should help |
|---|---|---|
| `loan_to_income` | Ratio | Loan size relative to capacity to pay |
| `installment_to_income` | Ratio | Monthly payment burden |
| `dti_tier` | Ordinal bin | Captures the non-linear DTI pattern from notebook 02 |
| `fico_avg` | Average of FICO range | Single number instead of high/low pair |
| `fico_band` | Ordinal bin | Captures FICO buckets used in industry |
| `credit_history_years` | Derived from `earliest_cr_line` | How long the borrower has had credit |
| `loan_per_year_history` | Ratio | Borrowing intensity relative to credit age |
| `is_long_term` | Binary | 60-month loans default differently from 36-month |

These run on raw values, before imputation, so they don't leak.

In [5]:
# CELL 10
# ─────────────────────────────────────────────────────────────────────────────
#  Engineered features  (computed on raw values, pre-split, no leakage)
# ─────────────────────────────────────────────────────────────────────────────

# (a) Loan-to-income ratio — capacity to pay
df['loan_to_income'] = df['loan_amnt'] / df['annual_inc'].replace(0, np.nan)

# (b) Installment-to-income — monthly payment burden (annualized)
df['installment_to_income'] = (df['installment'] * 12) / df['annual_inc'].replace(0, np.nan)

# (c) DTI tier — captures the non-linear default pattern from notebook 02
df['dti_tier'] = pd.cut(
    df['dti'],
    bins=[-np.inf, 10, 20, 30, 40, np.inf],
    labels=['<10', '10-20', '20-30', '30-40', '40+'],
)

# (d) FICO average — single number instead of high/low pair
df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2

# (e) FICO band — industry-standard credit tier groupings
df['fico_band'] = pd.cut(
    df['fico_avg'],
    bins=[0, 660, 700, 740, 780, 850],
    labels=['Subprime', 'Near-prime', 'Prime', 'Super-prime', 'Top-tier'],
)

# (f) Credit history length — years between first credit line and loan issue
df['credit_history_years'] = (
    (df['issue_d'] - df['earliest_cr_line']).dt.days / 365.25
).round(1)

# (g) Borrowing intensity — loans per year of credit history
df['loan_per_year_history'] = (
    df['loan_amnt'] / df['credit_history_years'].replace(0, np.nan)
)

# (h) Long-term flag — 60-month vs 36-month loans behave differently
df['is_long_term'] = (df['term'] == 60).astype(int)

# ─── Drop the raw columns we've now replaced or no longer need ──────────────
to_drop_after_engineering = [
    'fico_range_low', 'fico_range_high',   # replaced by fico_avg / fico_band
    'earliest_cr_line',                    # replaced by credit_history_years
    'issue_d',                             # already extracted issue_year
]
df = df.drop(columns=[c for c in to_drop_after_engineering if c in df.columns])

# ─── Sanity check on the new features ───────────────────────────────────────
new_features = [
    'loan_to_income', 'installment_to_income', 'dti_tier',
    'fico_avg', 'fico_band',
    'credit_history_years', 'loan_per_year_history', 'is_long_term',
]

check = pd.DataFrame({
    'feature':    new_features,
    'dtype':      [str(df[c].dtype) for c in new_features],
    'sample':     [df[c].dropna().iloc[0] for c in new_features],
    'missing %':  [(df[c].isna().mean() * 100).round(2) for c in new_features],
})

print("New engineered features:\n")
print(check.to_string(index=False))
print(f"\nFinal shape: {df.shape}")

New engineered features:

              feature    dtype   sample  missing %
       loan_to_income  float64     0.38       0.03
installment_to_income  float64     0.15       0.03
             dti_tier category    10-20       0.03
             fico_avg  float64   717.00       0.00
            fico_band category    Prime       0.00
 credit_history_years  float64    16.00       0.00
loan_per_year_history  float64 1,543.75       0.00
         is_long_term    int64        0       0.00

Final shape: (297299, 88)


# CELL 11
## 5. Train/test split

The split happens **before** any imputation or encoding. Computing fill values or categorical encodings on the full dataset would let test-set information leak into training, inflating model evaluation in notebook 04. By splitting first and fitting transformations on the training set only, we keep the test set genuinely held-out.

| Choice | Value | Reason |
|---|---|---|
| Test size | 20% (~60K loans) | Plenty for stable AUC estimates without starving training |
| Stratification | `default` × `issue_year` | Both classes and all vintages represented evenly in both sets |
| Random seed | 42 | Reproducibility |

Note on time-based splits: a forward-time holdout (train on 2008–2015, test on 2016–2018) would better simulate deployment, but adds complexity from the late-cycle vintage shift seen in notebook 02. Stratified random split is the standard choice for portfolio-grade demonstrations; the README will mention this trade-off.

In [6]:
# CELL 12
# ─────────────────────────────────────────────────────────────────────────────
#  Stratified train/test split  ·  stratify by default × issue_year
# ─────────────────────────────────────────────────────────────────────────────

# Build a composite stratification key so both default class and vintage
# year are balanced across train and test
strat_key = df['default'].astype(str) + '_' + df['issue_year'].astype(str)

# Drop rare strat-key combinations (fewer than 2 rows can't be stratified)
key_counts = strat_key.value_counts()
rare_keys = key_counts[key_counts < 2].index
if len(rare_keys) > 0:
    df = df[~strat_key.isin(rare_keys)].copy()
    strat_key = df['default'].astype(str) + '_' + df['issue_year'].astype(str)
    print(f"Dropped {len(rare_keys)} rare strat-key rows for clean stratification")

X = df.drop(columns=['default'])
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=strat_key,
    random_state=SEED,
)

# ─── Verify the split is balanced ───────────────────────────────────────────
print(f"Train shape : {X_train.shape}")
print(f"Test shape  : {X_test.shape}")
print(f"\nDefault rate balance:")
print(f"  Train : {y_train.mean():.2%}")
print(f"  Test  : {y_test.mean():.2%}")

print(f"\nVintage balance (default rate by year):")
balance = pd.DataFrame({
    'train_share': X_train.groupby('issue_year').size() / len(X_train),
    'test_share':  X_test.groupby('issue_year').size()  / len(X_test),
}).round(4) * 100
balance.columns = ['train %', 'test %']
print(balance)

Train shape : (237839, 87)
Test shape  : (59460, 87)

Default rate balance:
  Train : 19.96%
  Test  : 19.96%

Vintage balance (default rate by year):
            train %  test %
issue_year                 
2007           0.02    0.02
2008           0.11    0.11
2009           0.35    0.34
2010           0.86    0.86
2011           1.66    1.66
2012           3.99    3.99
2013           9.97    9.97
2014          16.62   16.62
2015          27.96   27.96
2016          21.73   21.74
2017          12.50   12.50
2018           4.23    4.23


# CELL 13
## 6. Imputation — train-fit, applied to both sets

Three categories of columns get different treatment:

| Column type | Strategy | Why |
|---|---|---|
| Numeric continuous | Fill with **training-set median** | Robust to outliers, doesn't shift distributions |
| Numeric semi-categorical (`emp_length`, `mort_acc`) | Fill with median + add `_was_missing` flag | Missingness itself may carry signal |
| Categorical | Fill with `'Unknown'` as its own level | Avoids forcing a guess; lets the model decide if missingness matters |

The fill values come from `X_train` only. The same values are then applied to `X_test`. This is the single most common spot for invisible data leakage in DS portfolios — and the reason notebook 04's evaluation will be honest rather than inflated.

In [7]:
# CELL 14
# ─────────────────────────────────────────────────────────────────────────────
#  Imputation  ·  fit values on X_train, apply to both X_train and X_test
# ─────────────────────────────────────────────────────────────────────────────

# Identify column types
numeric_cols = X_train.select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()
cat_cols     = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Columns where missingness might itself be informative — get a "was missing" flag
flag_cols = ['emp_length', 'mort_acc', 'pub_rec_bankruptcies']
flag_cols = [c for c in flag_cols if c in X_train.columns]

# (1) Add missing flags BEFORE imputing — capture the signal first
for col in flag_cols:
    flag_name = f'{col}_was_missing'
    X_train[flag_name] = X_train[col].isna().astype(int)
    X_test[flag_name]  = X_test[col].isna().astype(int)

# (2) Numeric columns — impute with training-set median
numeric_medians = X_train[numeric_cols].median()
X_train[numeric_cols] = X_train[numeric_cols].fillna(numeric_medians)
X_test[numeric_cols]  = X_test[numeric_cols].fillna(numeric_medians)

# (3) Categorical columns — fill with explicit 'Unknown' level
for col in cat_cols:
    if X_train[col].isna().any() or X_test[col].isna().any():
        # If the column is a pandas Categorical, we need to add 'Unknown' as a category
        if pd.api.types.is_categorical_dtype(X_train[col]):
            X_train[col] = X_train[col].cat.add_categories(['Unknown'])
            X_test[col]  = X_test[col].cat.add_categories(['Unknown'])
        X_train[col] = X_train[col].fillna('Unknown')
        X_test[col]  = X_test[col].fillna('Unknown')

# ─── Verify zero remaining missing values ───────────────────────────────────
train_missing = X_train.isna().sum().sum()
test_missing  = X_test.isna().sum().sum()

print(f"Remaining missing in X_train : {train_missing}")
print(f"Remaining missing in X_test  : {test_missing}")
print(f"\nFlags added                  : {flag_cols}")
print(f"\nFinal shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test : {X_test.shape}")

Remaining missing in X_train : 0
Remaining missing in X_test  : 0

Flags added                  : ['emp_length', 'mort_acc', 'pub_rec_bankruptcies']

Final shapes:
  X_train: (237839, 90)
  X_test : (59460, 90)


# CELL 15
## 7. Categorical encoding

Three encoding strategies depending on the variable:

| Variable type | Encoding | Why |
|---|---|---|
| Ordered categories (`grade`, `sub_grade`, `dti_tier`, `fico_band`) | Ordinal — preserve the natural ordering | The order itself is the signal |
| Unordered low-cardinality (`home_ownership`, `verification_status`, `purpose`, `application_type`, `initial_list_status`) | One-hot encoding | No artificial ordering imposed |
| High-cardinality (`addr_state` — 50 levels) | Target encoding fitted on training set only | Avoids exploding the feature count |

Target encoding for state replaces each state name with the *training-set* mean default rate for that state. This is leakage-prone if done naively (using the full dataset including test rows would be cheating), so the encoding map comes from `X_train` and `y_train` only, then applied to both splits.

In [8]:
# CELL 16
# ─────────────────────────────────────────────────────────────────────────────
#  Categorical encoding  ·  ordinal · one-hot · target (train-fit only)
# ─────────────────────────────────────────────────────────────────────────────

# (1) Ordinal encoding for naturally-ordered categories ─────────────────────
ordinal_maps = {
    'grade':     {g: i for i, g in enumerate(['A', 'B', 'C', 'D', 'E', 'F', 'G'])},
    'sub_grade': {f'{g}{n}': i for i, (g, n) in enumerate(
                    [(g, n) for g in 'ABCDEFG' for n in range(1, 6)])},
    'dti_tier':  {'<10': 0, '10-20': 1, '20-30': 2, '30-40': 3, '40+': 4, 'Unknown': -1},
    'fico_band': {'Subprime': 0, 'Near-prime': 1, 'Prime': 2, 'Super-prime': 3, 'Top-tier': 4, 'Unknown': -1},
}

for col, mapping in ordinal_maps.items():
    if col in X_train.columns:
        X_train[col] = X_train[col].map(mapping).astype(float)
        X_test[col]  = X_test[col].map(mapping).astype(float)

# (2) One-hot encoding for unordered low-cardinality variables ──────────────
one_hot_cols = [
    'home_ownership', 'verification_status', 'purpose',
    'application_type', 'initial_list_status', 'disbursement_method',
]
one_hot_cols = [c for c in one_hot_cols if c in X_train.columns]

# Use training-set categories as the canonical set; test set conforms
X_train_dummies = pd.get_dummies(X_train[one_hot_cols], drop_first=True, dtype=int)
X_test_dummies  = pd.get_dummies(X_test[one_hot_cols],  drop_first=True, dtype=int)

# Align columns — any test categories not seen in training get dropped,
# any training categories missing from test get filled with 0
X_test_dummies = X_test_dummies.reindex(columns=X_train_dummies.columns, fill_value=0)

X_train = pd.concat([X_train.drop(columns=one_hot_cols), X_train_dummies], axis=1)
X_test  = pd.concat([X_test.drop(columns=one_hot_cols),  X_test_dummies],  axis=1)

# (3) Target encoding for high-cardinality state variable ───────────────────
# Fit on training set only — for each state, compute the mean default rate
# in training data, with smoothing toward the global mean for small states
GLOBAL_RATE = y_train.mean()
SMOOTHING   = 100  # higher = more pull toward global mean for small states

state_counts = X_train.groupby('addr_state').size()
state_means  = X_train.assign(_y=y_train.values).groupby('addr_state')['_y'].mean()

state_target_map = (
    (state_means * state_counts + GLOBAL_RATE * SMOOTHING)
    / (state_counts + SMOOTHING)
).to_dict()

X_train['addr_state_te'] = X_train['addr_state'].map(state_target_map).fillna(GLOBAL_RATE)
X_test['addr_state_te']  = X_test['addr_state'].map(state_target_map).fillna(GLOBAL_RATE)

X_train = X_train.drop(columns=['addr_state'])
X_test  = X_test.drop(columns=['addr_state'])

# ─── Verify all columns are now numeric (model-ready) ───────────────────────
non_numeric_train = X_train.select_dtypes(exclude=['number']).columns.tolist()
non_numeric_test  = X_test.select_dtypes(exclude=['number']).columns.tolist()

print(f"Non-numeric columns in X_train: {len(non_numeric_train)}")
if non_numeric_train:
    print(f"  → {non_numeric_train}")

print(f"Non-numeric columns in X_test : {len(non_numeric_test)}")
if non_numeric_test:
    print(f"  → {non_numeric_test}")

print(f"\nFinal shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test : {X_test.shape}")

Non-numeric columns in X_train: 0
Non-numeric columns in X_test : 0

Final shapes:
  X_train: (237839, 107)
  X_test : (59460, 107)


# CELL 17
## 8. Persist splits for notebook 04

Saving as Parquet rather than CSV — Parquet preserves dtypes exactly (no string-to-int coercion surprises), compresses better, and loads ~10x faster. Notebook 04 starts by reading these four files and goes straight to model training.

| File | Contents |
|---|---|
| `outputs/X_train.parquet` | 237,839 rows × 107 features |
| `outputs/X_test.parquet`  | 59,460 rows × 107 features |
| `outputs/y_train.parquet` | Training labels |
| `outputs/y_test.parquet`  | Test labels |
| `outputs/feature_list.json` | Final feature names, for reference |

In [9]:
# CELL 18
# ─────────────────────────────────────────────────────────────────────────────
#  Persist the four splits + a feature manifest
# ─────────────────────────────────────────────────────────────────────────────

# Save the four splits
X_train.to_parquet('outputs/X_train.parquet', index=False)
X_test.to_parquet('outputs/X_test.parquet',   index=False)
y_train.to_frame('default').to_parquet('outputs/y_train.parquet', index=False)
y_test.to_frame('default').to_parquet('outputs/y_test.parquet',   index=False)

# Save the final feature list for documentation and notebook 04
with open('outputs/feature_list.json', 'w') as f:
    json.dump(
        {
            'n_features': X_train.shape[1],
            'features':   X_train.columns.tolist(),
        },
        f, indent=2,
    )

# ─── Confirm everything wrote correctly ─────────────────────────────────────
saved_files = [
    'outputs/X_train.parquet',
    'outputs/X_test.parquet',
    'outputs/y_train.parquet',
    'outputs/y_test.parquet',
    'outputs/feature_list.json',
]

print("Saved files:\n")
for f in saved_files:
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {f:<35}  {size_mb:>6.2f} MB")

print(f"\nFinal feature count: {X_train.shape[1]}")
print(f"Train: {X_train.shape[0]:,} rows  ·  Test: {X_test.shape[0]:,} rows  ·  Default rate: {y_train.mean():.2%}")

Saved files:

  outputs/X_train.parquet               23.96 MB
  outputs/X_test.parquet                 6.65 MB
  outputs/y_train.parquet                0.04 MB
  outputs/y_test.parquet                 0.01 MB
  outputs/feature_list.json              0.00 MB

Final feature count: 107
Train: 237,839 rows  ·  Test: 59,460 rows  ·  Default rate: 19.96%


# CELL 19
## 9. Handoff to notebook 04

The dataset is now model-ready. Every transformation in this notebook was justified, ordered to prevent leakage, and fit on the training set only.

### What this notebook produced

| Output | Used by |
|---|---|
| `outputs/X_train.parquet` | Notebook 04, model training |
| `outputs/X_test.parquet`  | Notebook 04, final evaluation |
| `outputs/y_train.parquet` | Notebook 04, training labels |
| `outputs/y_test.parquet`  | Notebook 04, evaluation labels |
| `outputs/feature_list.json` | Documentation, feature count |

### Engineering decisions, summarized

1. **Column drops were codified in notebook 02**, not invented here — every dropped column has a documented reason (sparsity, leakage, or no signal)
2. **Engineered features were computed on raw values pre-split** — they don't depend on imputation, so they don't leak
3. **Train/test split happened before imputation** — the test set never influenced the fill values
4. **Imputation was fit on training only** — medians for numeric, `'Unknown'` level for categorical, plus missingness flags where missingness might be informative
5. **Encoding was fit on training only** — ordinal where ordering exists, one-hot otherwise, target encoding with smoothing for high-cardinality state
6. **Stratified split balanced both default rate and vintage year** — both sets are statistically representative

### Next: notebook 04 — modeling

Notebook 04 trains three classifiers (Logistic Regression, Random Forest, XGBoost) on `X_train`, evaluates on the held-out `X_test`, compares them on AUC and SHAP-based explainability, and produces the `loan_predictions.csv` that notebook 05 turns into a dollar number.